# Sample procedure for analyzing an image

First, you must enter the path to the image (or generate a PSF) and the path to the folder result. (Here, we use the PSFGenerator to get a PSF image and image scale parameters, imread only return the PSF image)

In [ ]:
import os
from skimage.io import imread
from microscopy_metrics.scripts.PSFGenerator.PSF import PSFGenerator


ImagePath = ""  # Replace with the actual path to your image file
if not os.path.exists(ImagePath):
    psf = PSFGenerator()
else :
    psf = imread(ImagePath)

path = "" # Replace with the desired path to save the PSF image

The bead detection module identifies beads in microscopy images and extracts their regions of interest (ROIs). This step is required before calculating metrics or fitting PSFs.

In [ ]:
import numpy as np
from microscopy_metrics.detection import Detection
from microscopy_metrics.thresholdTools.threshold_tool import Threshold
from microscopy_metrics.detectionTools.detection_tool import DetectionTool

# Initialize the detector
detector = Detection()
detector.image = psf.psf  # Assign the PSF image

# Configure the detection tool (e.g., "Centroids", "peak local maxima", etc.)
detectionTool = DetectionTool.getInstance("Centroids")
detectionTool._image = psf.psf
detectionTool._thresholdTool = Threshold.getInstance("legacy")

# Assign the detection tool to the detector
detector._detectionTool = detectionTool

# Set detection parameters
detector.cropFactor = 5          # Crop factor for ROI extraction
detector.beadSize = 0.2         # Expected bead size (in micrometers)
detector.rejectionDistance = 0.5 # Minimum distance between beads (in micrometers)
detector.pixelSize = np.array([psf.dz, psf.dxy, psf.dxy])  # Pixel size (Z, Y, X)

# Run detection and save ROIs
for _ in detector.run(outputDir=path):
    pass

# Access the image analyzer for further processing
imageAnalyzer = detector._imageAnalyzer
imageAnalyzer._path = path

Before estimating the Full Width at Half Maximum (FWHM) of the PSF, you can compute theoretical metrics such as:

        Theoretical resolution (based on microscope parameters).

        Signal-to-background ratio (SBR).



In [ ]:
import os
from microscopy_metrics.metrics import Metrics
from microscopy_metrics.resolutionTools.theoretical_resolution import TheoreticalResolution

# Initialize the metrics tool
MetricTool = Metrics()
MetricTool._imageAnalyzer = imageAnalyzer

# Configure theoretical resolution parameters
MetricTool._TheoreticalResolutionTool = TheoreticalResolution.getInstance("widefield")
MetricTool._TheoreticalResolutionTool._numericalAperture = psf.NA
MetricTool._TheoreticalResolutionTool._emissionWavelength = psf.wvl
MetricTool._TheoreticalResolutionTool._refractiveIndex = psf.ni
MetricTool._TheoreticalResolutionTool._excitationWavelength = 0.225  # Default excitation wavelength

# Set ring parameters for SBR calculation
MetricTool._ringInnerDistance = 1.0  # Inner radius for background estimation
MetricTool._ringThickness = 2.0      # Thickness of the background ring

# Run pre-fitting metrics
for _ in MetricTool.runPrefittingMetrics():
    pass

# Save mesh data for each bead (optional)
for bead in imageAnalyzer._beadAnalyzer:
    if bead._rejected == False and bead._roi is not None and bead._metricTool.meshBuilder is not None:
        bead._metricTool.meshBuilder.saveMesh(os.path.join(path, f"bead_{bead._id}_mesh.obj"))

Estimate the Full Width at Half Maximum (FWHM) of the PSF by fitting a Gaussian function to each detected bead. This step is crucial for characterizing the resolution of your microscope.

In [ ]:
from microscopy_metrics.fitting import Fitting

# Initialize the fitting tool
FittingTool = Fitting()
FittingTool._imageAnalyzer = imageAnalyzer

# Configure fitting parameters
FittingTool.fitType = "2D"  # or "1D", "3D", "2D Ellips", "2D Rotation", "3D Rotation"
FittingTool._thresholdRSquared = 0.5  # Minimum R² value for a valid fit

# Run the fitting
FittingTool.computeFitting()

After fitting, compute advanced metrics such as:

        Asymmetry (deviation from a perfect Gaussian).

        Aberration metrics (e.g., astigmatism, coma).



In [ ]:
# Assign the fitting results to the metrics tool
MetricTool._imageAnalyzer = FittingTool._imageAnalyzer

# Run metrics calculation
for _ in MetricTool.runMetrics():
    pass

# Access the updated image analyzer
imageAnalyzer = MetricTool._imageAnalyzer

Generate reports summarizing the analysis results, including:

        Acquisition parameters (e.g., microscope settings).

        Detected beads (positions, intensities).

        Fitted PSFs (FWHM, R² values).

        Metrics (resolution, SNR, asymmetry).

        Visualizations (heatmaps, fitting curves).



In [ ]:
from microscopy_metrics.report_generator import ReportGenerator

# Crop PSF images for the report
detector.cropPsf(path)          # Crop individual PSFs
detector.GlobalCropPsf(path)    # Crop global PSF overview

# Generate heatmaps
MetricTool.GenerateHeatmap(path)

# Display fitting results
FittingTool.displayFitting(path)

# Generate the HTML report
ReportGenerator = ReportGenerator().getInstance("HTML")
ReportGenerator._inputDir = path
ReportGenerator._imageAnalyzer = imageAnalyzer
ReportGenerator.generateReport(path)